# Generate, exchange and refine material-marked meshes

Build a unit square with two materials separated at x=0.5, mark its inlet,
outlet and internal interface, write a real VTU exchange file, and transfer
labels through exact nested refinement. Labels remain separate from PDE data.

Run in the locked `test` environment (Gmsh, meshio, Matplotlib). This notebook
uses package mesh/ancestry owners; it does not implement another mesher.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/4bbda3f89e2613048c8cfb6dba61febad08ced801fe225f9831df7e614c4630e/marked_materials-companion.zip"
COMPANION_SHA256 = "4bbda3f89e2613048c8cfb6dba61febad08ced801fe225f9831df7e614c4630e"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("foundations/geometry/marked_materials.ipynb", directory=ROOT)


In [ ]:
from pathlib import Path
import os

ROOT = Path(os.environ.get("PYMHM_WORKSPACE", Path.cwd())).resolve()
OUTPUT = ROOT / "build" / "guide-meshing"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT)


## 1. Generate two surfaces sharing the same interface curve


In [ ]:
import gmsh
from pymhm.io.planar import from_gmsh

gmsh.initialize()
try:
    gmsh.model.add("two-materials")
    coordinates = [(0, 0), (0.5, 0), (1, 0), (1, 1), (0.5, 1), (0, 1)]
    points = [gmsh.model.geo.addPoint(x, y, 0, 0.15) for x, y in coordinates]
    edges = [
        gmsh.model.geo.addLine(points[a], points[b])
        for a, b in [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 0), (1, 4)]
    ]
    left_loop = gmsh.model.geo.addCurveLoop([edges[i] for i in [0, 6, 4, 5]])
    right_loop = gmsh.model.geo.addCurveLoop([edges[1], edges[2], edges[3], -edges[6]])
    left = gmsh.model.geo.addPlaneSurface([left_loop])
    right = gmsh.model.geo.addPlaneSurface([right_loop])
    gmsh.model.geo.synchronize()
    for tag, surface, name in [(1, left, "matrix"), (2, right, "channel")]:
        gmsh.model.addPhysicalGroup(2, [surface], tag)
        gmsh.model.setPhysicalName(2, tag, name)
    for tag, edge, name in [(10, edges[5], "inlet"), (20, edges[2], "outlet"),
                            (30, edges[6], "material interface")]:
        gmsh.model.addPhysicalGroup(1, [edge], tag)
        gmsh.model.setPhysicalName(1, tag, name)
    gmsh.model.mesh.generate(2)
    data = from_gmsh()
finally:
    gmsh.finalize()


## 2. Inspect material, exterior boundary and interior face IDs


In [ ]:
import numpy as np

assert set(np.unique(data.cell_tags)) == {1, 2}
inlet = data.boundary_tags[10]
outlet = data.boundary_tags[20]
interface = data.face_tags[30]
assert not set(interface).intersection(data.mesh.boundary_faces)

permeability_by_id = {1: 1.0, 2: 100.0}
cell_permeability = np.array([permeability_by_id[int(tag)] for tag in data.cell_tags])


## 3. Export a lossless exchange file

VTU retains physical IDs/names, internal and exterior face groups, and attached
cell permeability. Tags do not automatically impose boundary data.


In [ ]:
from pymhm.io.planar import read_mesh, write_mesh

write_mesh(OUTPUT / "materials.vtu", data, cell_data={"permeability": cell_permeability})
restored = read_mesh(OUTPUT / "materials.vtu")
assert restored.physical_names == data.physical_names
assert restored.face_tags == data.face_tags
np.testing.assert_array_equal(restored.cell_tags, data.cell_tags)
np.testing.assert_array_equal(restored.cell_data["permeability"], cell_permeability)


## 4. Refine using exact ancestry

Cell ancestry transfers material IDs. Face ancestry transfers the original
material-interface group. A nonnested remesher requires the physical labels
to be reapplied from the geometry/material description.


In [ ]:
from pymhm.meshes.refinement import refine_triangles

refinement = refine_triangles(data.mesh, marked=data.cell_tags == 1)
new_material_ids = data.cell_tags[refinement.parent_cells]
new_interface_faces = np.flatnonzero(
    np.isin(refinement.parent_faces, data.face_tags[30])
)


## 5. Plot both meshes and the preserved interface

The material colors use the same IDs. Highlight the original macro mesh and the
actual marked interface; new fine interior edges do not become material faces.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import BoundaryNorm

figure, axes = plt.subplots(1, 2, figsize=(11, 4.8), constrained_layout=True)
for ax, mesh, tags, interface_faces, title in [
    (axes[0], data.mesh, data.cell_tags, data.face_tags[30], "Original marked macro mesh"),
    (axes[1], refinement.mesh, new_material_ids, new_interface_faces, "Refined mesh and inherited labels"),
]:
    plot = ax.tripcolor(
        mesh.points[:,0], mesh.points[:,1], mesh.cells, facecolors=tags,
        cmap="viridis", norm=BoundaryNorm([0.5, 1.5, 2.5], 256), edgecolors="white", linewidth=0.25,
    )
    ax.add_collection(LineCollection(data.mesh.points[data.mesh.faces], colors="black", linewidths=0.3, alpha=0.45))
    ax.add_collection(LineCollection(mesh.points[mesh.faces[list(interface_faces)]], colors="red", linewidths=2))
    ax.set(title=title, xlabel="x", ylabel="y", aspect="equal")
    figure.colorbar(plot, ax=ax, ticks=[1,2], label="Material ID", shrink=0.8)
figure.savefig(OUTPUT / "marked-materials.png", dpi=180)
plt.show()


## 6. Connect the tags to a PDE provider

Use `permeability_by_id` to choose each macrocell's coefficient inside its local
UFL callback. If a material jump crosses a macrocell, keep its fine material
partition and integrate the jump there. A macrocell average changes the problem.

Select external boundary conditions using `boundary_tags`; the internal group
in `face_tags` remains coupled to both incident cells. The canonical face
orientation comes from the mesh, not from the generator's curve ordering.

For tetrahedra, prisms, hexahedra and polyhedra, use `VolumeMeshData` and the
companion three-dimensional mesh-exchange notebook. The volume API preserves
integer material/face IDs, with its own connectivity conventions.

## References

- Gmsh reference manual: https://gmsh.info/doc/texinfo/
- meshio project: https://github.com/nschloe/meshio
